## Import Dependencies

In [41]:
from transformers import AutoTokenizer
import polars as pl
from datasets import Dataset

## Import Data

In [44]:
train_path = "/content/complete_train.parquet"
test_path = "/content/complete_test.parquet"

In [45]:
train_df = pl.scan_parquet(train_path)
test_df = pl.scan_parquet(test_path)

In [46]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length
str,str,str,i64,str,str,str,str,i64
"""3VKW""","""A""","""SER""",1,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""TYR""",2,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""THR""",3,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""ARG""",4,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""SER""",5,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
…,…,…,…,…,…,…,…,…
"""5XVT""","""A""","""LEU""",693,"""A""","""T""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""ARG""",694,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""SER""",695,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697


## Process Data

### Add Labels

In [47]:
train_df = (train_df
            .with_columns(label = pl.col('secondary_structure').rank(method='dense') - 1)
              .cast({'label': pl.Int64}))

test_df = (test_df
            .with_columns(label = pl.col('secondary_structure').rank(method='dense') - 1)
              .cast({'label': pl.Int64}))


### Removing Duplicates on Index
Some indices have two amino acids, so remove the one without a secondary structure (label = 0)

In [52]:
train_removed = (
    train_df.sort(['id', 'asym_id', 'index', 'label'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'label'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                label = pl.col('label').list.slice(1),
            )
            .explode(['amino_acid', 'label'])
)

train_df = (
    train_df.join(train_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         label = pl.when(~(pl.col('label_right').is_null()))
                                        .then(pl.col('label_right'))
                                        .otherwise(pl.col('label'))
            )
            .drop(['amino_acid_right', 'label_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)


In [53]:
test_removed = (
    test_df.sort(['id', 'asym_id', 'index', 'label'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'label'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                label = pl.col('label').list.slice(1),
            )
            .explode(['amino_acid', 'label'])
)

test_df = (
    test_df.join(test_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         label = pl.when(~(pl.col('label_right').is_null()))
                                        .then(pl.col('label_right'))
                                        .otherwise(pl.col('label'))
            )
            .drop(['amino_acid_right', 'label_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)


In [54]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length,label
str,str,str,i64,str,str,str,str,i64,i64
"""1A1X""","""A""","""GLY""",1,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,0
"""1A1X""","""A""","""SER""",2,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,0
"""1A1X""","""A""","""ALA""",3,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
"""1A1X""","""A""","""GLY""",4,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
"""1A1X""","""A""","""GLU""",5,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
…,…,…,…,…,…,…,…,…,…
"""7ODC""","""A""","""ILE""",420,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0
"""7ODC""","""A""","""GLN""",421,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0
"""7ODC""","""A""","""SER""",422,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0


#### Remove Duplicates on Asym ID
Some sequences are exactly the same and have the same secondary structures, so choose only one Asym ID (first ID)

In [55]:
train_df = (

  (train_df.sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
                .agg(['label', 'index'])

  ).sort('asym_id')
    .group_by(['id', 'sequence', 'label', 'index'])
      .agg(['asym_id'])
        .with_columns(asym_id = pl.col('asym_id').list.slice(0,1))
          .explode(['asym_id'])
            .explode(['label', 'index'])

)

test_df = (

  (test_df.sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
                .agg(['label', 'index'])

  ).sort('asym_id')
    .group_by(['id', 'sequence', 'label', 'index'])
      .agg(['asym_id'])
        .with_columns(asym_id = pl.col('asym_id').list.slice(0,1))
          .explode(['asym_id'])
            .explode(['label', 'index'])

)

In [56]:
test_df.collect()

id,sequence,label,index,asym_id
str,str,i64,i64,str
"""5CVW""","""GSARDDVLIGDAGANVLNGLAGNDVLSGGA…",1,1,"""A"""
"""5CVW""","""GSARDDVLIGDAGANVLNGLAGNDVLSGGA…",1,2,"""A"""
"""5CVW""","""GSARDDVLIGDAGANVLNGLAGNDVLSGGA…",8,3,"""A"""
"""5CVW""","""GSARDDVLIGDAGANVLNGLAGNDVLSGGA…",8,4,"""A"""
"""5CVW""","""GSARDDVLIGDAGANVLNGLAGNDVLSGGA…",1,5,"""A"""
…,…,…,…,…
"""2UX1""","""GSPAEIASFSPRPSLADSKAVLNQAVADLS…",1,161,"""I"""
"""2UX1""","""GSPAEIASFSPRPSLADSKAVLNQAVADLS…",1,162,"""I"""
"""2UX1""","""GSPAEIASFSPRPSLADSKAVLNQAVADLS…",1,163,"""I"""


### Prepare for Tokenization

In [57]:
train_agg = (train_df
              .select(['id', 'asym_id', 'sequence', 'index', 'label'])
              .sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['index', 'label'])
            )

test_agg = (test_df
              .select(['id', 'asym_id', 'sequence', 'index', 'label'])
              .sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['index', 'label'])
            )

In [59]:
train_ds = Dataset.from_polars(train_agg.collect())
test_ds = Dataset.from_polars(test_agg.collect())

### Tokenization

In [61]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")

def tokenize_and_label(examples):
  tokenized_inputs = tokenizer(examples["sequence"],
                                return_tensors="pt",
                                add_special_tokens=False,
                                padding=False)
  tokenized_inputs["input_ids"] = tokenized_inputs["input_ids"][0]
  tokenized_inputs["attention_mask"] = tokenized_inputs["attention_mask"][0]
  return tokenized_inputs

In [62]:
tokenized_train = train_ds.map(tokenize_and_label, batched=False)
tokenized_test = test_ds.map(tokenize_and_label, batched=False)

Map:   0%|          | 0/11425 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

## Output Data

### Dataset to DataFrame

In [63]:
output_train = Dataset.to_polars(tokenized_train).lazy()
output_test = Dataset.to_polars(tokenized_test).lazy()

In [64]:
output_train = (output_train
                .join(train_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'label'],
                      how='inner')
                .explode(['index', 'label', 'input_ids', 'attention_mask'])
                )

output_test = (output_test
                .join(test_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'label'],
                      how='inner')
                .explode(['index', 'label', 'input_ids', 'attention_mask'])
                )


In [65]:
output_test.collect()

id,asym_id,sequence,index,label,input_ids,attention_mask
str,str,str,i64,i64,i32,i8
"""3A5P""","""A""","""MVWSVQIVDNAGLGANLALYPSGNSSTVPR…",1,0,20,1
"""3A5P""","""A""","""MVWSVQIVDNAGLGANLALYPSGNSSTVPR…",2,1,7,1
"""3A5P""","""A""","""MVWSVQIVDNAGLGANLALYPSGNSSTVPR…",3,3,22,1
"""3A5P""","""A""","""MVWSVQIVDNAGLGANLALYPSGNSSTVPR…",4,3,8,1
"""3A5P""","""A""","""MVWSVQIVDNAGLGANLALYPSGNSSTVPR…",5,3,7,1
…,…,…,…,…,…,…
"""5BY7""","""C""","""MAHHHHHHSSGLEVLFQGPMTRTVSVLSAA…",356,3,5,1
"""5BY7""","""C""","""MAHHHHHHSSGLEVLFQGPMTRTVSVLSAA…",357,3,4,1
"""5BY7""","""C""","""MAHHHHHHSSGLEVLFQGPMTRTVSVLSAA…",358,3,9,1


### Write to Parquet

In [66]:
output_train.collect().write_parquet('tokenized_train.parquet')
output_test.collect().write_parquet('tokenized_test.parquet')

#### Replacing Labels

In [67]:
output_train = output_train.with_columns(
                  label=pl.when(pl.col('label')==0)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_train.collect().write_parquet('tokenized_train_c9.parquet')

output_test = output_test.with_columns(
                  label=pl.when(pl.col('label')==0)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_test.collect().write_parquet('tokenized_test_c9.parquet')


In [68]:
output_train = output_train.with_columns(
                  label=pl.when(pl.col('label')==1)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_train.collect().write_parquet('tokenized_train_c8.parquet')

output_test = output_test.with_columns(
                  label=pl.when(pl.col('label')==1)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_test.collect().write_parquet('tokenized_test_c8.parquet')
